In [1]:
meta_file_path = '/storage/riosugimuralab/alexto/A3_VisiumHD/cellphonefiles/metadata.tsv'
counts_file_path = '/storage/riosugimuralab/alexto/A3_VisiumHD/cellphonefiles/normalised_log_counts.h5ad'
cpdb_file_path = '/storage/riosugimuralab/alexto/Z2_cpdb/v5.0.0/cellphonedb.zip'
out_path = '/storage/riosugimuralab/alexto/A3_VisiumHD/cellphonefiles/'

In [2]:
import os
import pandas as pd

out_path = "/storage/riosugimuralab/alexto/A3_VisiumHD/cellphonefiles/"  # absolute dir [web:35][web:36]

scores_file = os.path.join(out_path, "simple_analysis_interaction_scores_10_12_2025_210538.txt")  # [web:35][web:38]

scores = pd.read_csv(
    scores_file,
    sep="\t",
    header=0
)

meta_cols = [
    "id_cp_interaction",
    "interacting_pair",
    "gene_a",
    "gene_b",
    "partner_a",
    "partner_b"
]

cell_types_of_interest = ["Mac_CD16", "Mac_IL1B", "Mac_GLUL", "Tumor", "Fibro", "HEC"]

pair_cols = [c for c in scores.columns if "|" in c]

selected_pair_cols = []
for c in pair_cols:
    ct1, ct2 = c.split("|")
    if ct1 in cell_types_of_interest and ct2 in cell_types_of_interest:
        selected_pair_cols.append(c)

long_scores = scores.melt(
    id_vars=[c for c in meta_cols if c in scores.columns],
    value_vars=selected_pair_cols,
    var_name="cell_pair",
    value_name="score"
)

long_scores = long_scores.dropna(subset=["score"])

top5 = long_scores.sort_values("score", ascending=False).head(5)

top5[["interacting_pair", "cell_pair", "score"]]


,interacting_pair,cell_pair,score
69617,VEGFA_KDR,Tumor|Fibro,100.0
81106,WNT7B_FZD4_LRP5,Tumor|Tumor,100.0
3783,PGD2_byPTGDS_PTGDR2,Fibro|HEC,100.0
81111,NDP_FZD4_LRP5,Tumor|Tumor,100.0
68051,COL8A1_integrin_a11b1_complex,Tumor|Fibro,100.0


In [3]:
top5_per_pair = (
    long_scores
    .sort_values(["cell_pair", "score"], ascending=[True, False])
    .groupby("cell_pair")
    .head(5)
)

top5_per_pair[["cell_pair", "interacting_pair", "score"]]


,cell_pair,interacting_pair,score
49,Fibro|Fibro,CDH7_CDH7,100.0
74,Fibro|Fibro,COL11A1_integrin_a1b1_complex,100.0
77,Fibro|Fibro,COL13A1_integrin_a1b1_complex,100.0
78,Fibro|Fibro,COL14A1_integrin_a1b1_complex,100.0
79,Fibro|Fibro,COL15A1_integrin_a1b1_complex,100.0
...,...,...,...
79232,Tumor|Tumor,CEACAM1_CEACAM6,100.0
79233,Tumor|Tumor,CEACAM5_CEACAM6,100.0
79234,Tumor|Tumor,CEACAM6_CEACAM6,100.0
79243,Tumor|Tumor,CEACAM5_CEACAM1,100.0


In [4]:
# Only rows where cell_pair begins with "Mac_IL1B|" (no Fibro|Mac_IL1B, etc.) [web:53][web:56]
mac_il1b_scores = long_scores[long_scores["cell_pair"].str.startswith("Mac_IL1B|", na=False)]

# The specific Mac_IL1B→X pairs present [web:53][web:56]
mac_il1b_pairs = mac_il1b_scores["cell_pair"].unique()
print(mac_il1b_pairs)


['Mac_IL1B|Fibro' 'Mac_IL1B|HEC' 'Mac_IL1B|Mac_CD16' 'Mac_IL1B|Mac_GLUL'
 'Mac_IL1B|Mac_IL1B' 'Mac_IL1B|Tumor']


In [5]:
cell_types_of_interest = ["Mac_CD16", "Mac_IL1B", "Mac_GLUL", "Tumor", "Fibro", "HEC"]

for sender in cell_types_of_interest:
    # sender|X only
    df_sender = long_scores[long_scores["cell_pair"].str.startswith(f"{sender}|", na=False)]  # [web:56][web:92]

    for pair, df_pair in df_sender.groupby("cell_pair"):  # [web:90][web:95]
        top = df_pair.nlargest(10, "score")[["interacting_pair", "cell_pair", "score"]]  # [web:83][web:89]

        print("#" * 80)
        print(f"Sender: {sender}  |  Pair: {pair}")
        print(top)


################################################################################
Sender: Mac_CD16  |  Pair: Mac_CD16|Fibro
                      interacting_pair       cell_pair   score
28658  ProstaglandinF2a_byAKR1B1_PTGFR  Mac_CD16|Fibro  90.884
28564                      DLL1_NOTCH2  Mac_CD16|Fibro  87.480
27576                     APLP2_PLXNA4  Mac_CD16|Fibro  87.394
27960                       NRG1_ERBB4  Mac_CD16|Fibro  79.473
28609                       VEGFB_FLT1  Mac_CD16|Fibro  71.493
28612               VEGFB_FLT1_complex  Mac_CD16|Fibro  71.493
28866                     TGFB1_TGFBR3  Mac_CD16|Fibro  67.024
27959                      HBEGF_ERBB4  Mac_CD16|Fibro  65.939
27575                       APP_PLXNA4  Mac_CD16|Fibro  57.914
28859                         HFE_TFRC  Mac_CD16|Fibro  56.296
################################################################################
Sender: Mac_CD16  |  Pair: Mac_CD16|HEC
                         interacting_pair     cell_pair   score